## Tool definition

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.tools import tool

# we can either define our tool using the @tool decorator
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
# or like this
# the function name is the tool what our agent is going to use
# we can either add the doc string inside the function

@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [5]:
# or in the tool decorator as tool description

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [6]:
# this is exactly what our agent is going to do when using the tools it has
# see, we use the function name to invoke the tool. not the tool name (square root)

tool1.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [8]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately 21.61018278497431.  
(It’s an irrational number, so it’s often rounded to 21.610183 or 21.610.)


In [ ]:
from pprint import pprint

pprint(response['messages'])

# something important to notice in this output
# first we see our HumanMessage asking about the square root of 467
# then we see the AIMessage but with content='' 
# but if we scroll ahead, we see this in the AIMessage: tool_calls=[{'name': 'square_root', 'args': {'x': 467}
# then we see the ToolMessage containing the content: content='21.61018278497431'
# in the end we the AIMessage that tells us the square root of 467 in natural language

# so what happened is, the model got the query and looked at its tool and decided which tool to call with the correct args
# then the tool actually called. the tool result then went back to the model which then gave us the answer in NL

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='767272fe-b4ff-475f-8632-8b0853b4b307'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2007, 'prompt_tokens': 158, 'total_tokens': 2165, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1984, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8OOPTwj2NnAcwQ6DFCnBe4HiJcfx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc22c-14e9-7f50-bd36-9d3a0f63f04a-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_67H60jn4HHj4xNpNClCNziey', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 2007, 'total

In [ ]:
print(response["messages"][1].tool_calls)